In [ ]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np
import json
import requests
import time
import threading
import os

MODEL_PATH = "./wild_animal_model.keras"
LABEL_PATH = "./class_labels.json"

model = load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_labels = json.load(f)

class_labels = {v: k for k, v in class_labels.items()}

print("Model loaded successfully!")

last_alert_time = 0
ALERT_INTERVAL = 10   # seconds
last_animal = None

def send_alert(animal, image_filename):
    try:
        requests.post(
            "http://localhost:5000/detect",
            json={
                "animal": animal,
                "location": "Camera 1",
                "imageUrl": f"/detections_images/{image_filename}"
            }
        )
        print("Alert sent:", animal)
    except Exception as e:
        print("Server error:", e)

if not os.path.exists("detections"):
    os.makedirs("detections")

cap = None

for i in range(5):
    temp_cap = cv2.VideoCapture(i)
    if temp_cap.isOpened():
        print(f"Camera detected at index {i}")
        cap = temp_cap
        break

if cap is None:
    print("No camera detected.")
    exit()

print("Camera started!")

while True:

    ret, frame = cap.read()

    if not ret:
        print("Frame error")
        break

    # Preprocess
    img = cv2.resize(frame, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    prediction = model.predict(img, verbose=0)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction)

    label = "No Animal"
    animal = "No Animal"

    if confidence > 0.80:
        animal = class_labels[class_index]
        label = f"{animal} ({confidence:.2f})"

    current_time = time.time()

    if animal != "No Animal" and animal!= "Monkey":
        if animal != last_animal and (current_time - last_alert_time > ALERT_INTERVAL):
            print("New Detection:", animal)
            cv2.imwrite("latest.jpg", frame)
            timestamp = int(current_time)
            filename = f"{animal}_{timestamp}.jpg"
            filepath = f"detections/{filename}"
            cv2.imwrite(filepath, frame)
            threading.Thread(target=send_alert, args=(animal, filename)).start()
            last_alert_time = current_time
            last_animal = animal
        elif (current_time - last_alert_time > 60):
            animal="No Animal"
    # Draw label
    cv2.putText(
        frame,
        label,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # Show video
    cv2.imshow("Wild Animal Detection", frame)

    time.sleep(0.1)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [ ]:
if 'cap' in globals() and cap is not None:
    cap.release()
cv2.destroyAllWindows()